# Phase 1: Tokenizer audit

**Project.** Off-Manifold Failure (BlackboxNLP 2026 archival).

**Goal of this notebook.** For each of the three target models (GPT-J 6B, Pythia 6.9B, Llama 3.1 8B), determine which `(a, b) in {0,...,99}^2` pairs have:
- single-token operand `a`
- single-token operand `b`
- single-token answer `s = a + b`

Then compute the three-way intersection (pairs retained by every model) and save to Drive.

**Why this is Phase 1.** Defining the analysis dataset is a prerequisite for every later phase. See [colab_execution_plan.md section 5](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/colab_execution_plan.md) for full context, and [full_paper_plan.md section 3.2](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/full_paper_plan.md) for the methodology.

**Runtime.** CPU is sufficient (no model load needed for tokenizer ops). Estimated wall time: <= 10 minutes.

**Inputs.** None on Drive (this is the first notebook).

**Outputs to Drive.**
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/gpt-j-6b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/pythia-6.9b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/llama-3.1-8b.json`
- `/MyDrive/blackbox_nlp_2026/tokenizer_audit/intersection.json`

## 1. Standard prelude (Drive mount, repo clone, dependency install, HF login)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/blackbox_nlp_2026'
CODE_DIR = f'{PROJECT}/code_repo'
REPO_URL = 'https://github.com/anshulk-cmu/blackbox-nlp-2026.git'

os.makedirs(PROJECT, exist_ok=True)
if not os.path.exists(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}
!git pull --ff-only

In [ ]:
# Pin dependency versions per colab_execution_plan.md section 3.
# Phase 1 only needs transformers (for tokenizers); torch is already on Colab.
!pip install -q \
    transformers==4.45.0 \
    huggingface_hub==0.25.1

In [ ]:
# HuggingFace login. HF_TOKEN must be stored in Colab Secrets (key icon in sidebar).
# DO NOT paste the token in this notebook or any committed file.
from google.colab import userdata
import huggingface_hub
hf_token = userdata.get('HF_TOKEN')
assert hf_token is not None and hf_token.startswith('hf_'), \
    'HF_TOKEN not set in Colab Secrets. Add it via the key icon in the sidebar.'
huggingface_hub.login(hf_token)
print('HF login OK')

In [ ]:
# Confirm runtime (Phase 1 does not need a GPU; this is just informational).
import torch, sys
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 2. Load all three tokenizers

We do **not** load the model weights here -- only the tokenizers, which are tiny (a few MB each).

In [ ]:
from transformers import AutoTokenizer

MODELS = {
    'gpt-j-6b':     'EleutherAI/gpt-j-6B',
    'pythia-6.9b':  'EleutherAI/pythia-6.9b',
    'llama-3.1-8b': 'meta-llama/Llama-3.1-8B',
}

tokenizers = {}
for key, hf_name in MODELS.items():
    print(f'Loading tokenizer for {key} ({hf_name})...')
    tokenizers[key] = AutoTokenizer.from_pretrained(hf_name)
    print(f'  vocab size: {tokenizers[key].vocab_size}')
print('All tokenizers loaded.')

## 3. Run the audit per model

Imports `code/tokenizer_audit.py` (already cloned with the repo) and applies it to each tokenizer. The audit logic is the same code path the local self-test exercises -- see [code/tokenizer_audit.py](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/code/tokenizer_audit.py) for the implementation.

In [ ]:
# Insert code/ directly (NOT the repo root) to avoid shadowing Python's
# built-in `code` module.
sys.path.insert(0, f'{CODE_DIR}/code')
from tokenizer_audit import (
    audit_tokenizer, intersect_audits, save_audit, save_intersection,
    summary_report, PROMPT_TEMPLATES,
)

AUDIT_DIR = f'{PROJECT}/tokenizer_audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

audits = []
for key in MODELS:
    print(f'\nAuditing {key}...')
    print(f'  prompt template: {PROMPT_TEMPLATES[key]!r}')
    audit = audit_tokenizer(tokenizers[key], key)
    audits.append(audit)
    out_path = f'{AUDIT_DIR}/{key}.json'
    save_audit(audit, out_path)
    print(f'  retained {audit["n_retained"]} / 10000, dropped {audit["n_dropped"]}.')
    print(f'  saved to {out_path}')

## 4. Three-way intersection

In [ ]:
intersection = intersect_audits(audits)
intersect_path = f'{AUDIT_DIR}/intersection.json'
save_intersection(intersection, audits, intersect_path)
print(f'Saved {len(intersection)} retained pairs to {intersect_path}')

## 5. Summary report

In [ ]:
print(summary_report(audits, intersection))

## 6. Sanity assertion (pre-registered gate)

Per [colab_execution_plan.md section 5](https://github.com/anshulk-cmu/blackbox-nlp-2026/blob/main/colab_execution_plan.md), the three-way intersection must contain at least 7000 pairs. If less, investigate before proceeding to Phase 2.

In [ ]:
n_int = len(intersection)
if n_int < 7000:
    print(f'WARNING: intersection has only {n_int} pairs (target >= 7000).')
    print('Investigate per-model dropped lists before continuing to Phase 2.')
    for au in audits:
        bad_ops = [n for n, ok in au['operand_ok'].items() if not ok]
        bad_ans = [n for n, ok in au['answer_ok'].items() if not ok]
        print(f'  {au["model"]}: multi-token operands={bad_ops!r}, multi-token answers={bad_ans!r}')
else:
    print(f'PASS: intersection has {n_int} pairs (>= 7000 target).')
    print('Proceed to Phase 2 (accuracy reproduction).')

## 7. Optional: per-tokenizer diagnostic

If any model dropped pairs, the cell below shows which integer values in 0..198 are multi-token. This is the most common reason for drops (e.g., some BPE tokenizers split certain three-digit numbers like 100, 200).

In [ ]:
for au in audits:
    bad_ans_ints = sorted(int(n) for n, ok in au['answer_ok'].items() if not ok)
    bad_op_ints = sorted(int(n) for n, ok in au['operand_ok'].items() if not ok)
    print(f'\n{au["model"]}:')
    print(f'  Multi-token operands (0..99):  {bad_op_ints if bad_op_ints else "(none)"}')
    print(f'  Multi-token answers (0..198):  {bad_ans_ints if bad_ans_ints else "(none)"}')

## 8. Done

Phase 1 outputs are now in `/MyDrive/blackbox_nlp_2026/tokenizer_audit/`. Proceed to Phase 2 (accuracy reproduction) once:

1. The intersection size is comfortable (>= 7000).
2. The per-model retained counts roughly match KT's reported coverage.

Phase 2 lives in `notebooks/02a_accuracy_gptj.ipynb`, `02b_accuracy_pythia.ipynb`, `02c_accuracy_llama.ipynb` (to be written next).